MODELO ORIGINAL DA FASE 1

In [ ]:
"""
Tech Challenge - Fase 1 - Challenge B

Script unico para executar o pipeline completo pedido no PDF:
- carregamento e discussao do dataset;
- exploracao, estatisticas descritivas e visualizacoes;
- limpeza e pre-processamento;
- analise de correlacao;
- treino, validacao e teste com multiplos modelos de classificacao;
- avaliacao com accuracy, recall e F1-score;
- interpretacao com feature importance e SHAP quando disponivel;
- geracao de artefatos, relatorio, README e Dockerfile.

Uso:
    python techchallengeB.py
    python techchallengeB.py --data "C:/caminho/para/data.csv" --output outputs_techchallenge_b
"""

from __future__ import annotations

import argparse
import json
import math
import pickle
import textwrap
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Configuracoes centrais do problema: coluna alvo original, alvo binario
# criado pelo script e codigos usados para as classes.
TARGET_COLUMN = "Dangerous"
TARGET_NAME = "dangerous_binary"
POSITIVE_LABEL = 1
NEGATIVE_LABEL = 0

# Caminhos testados automaticamente quando o usuario nao informa --data.
DEFAULT_DATASET_CANDIDATES = [
    Path("data.csv"),
    Path(r"/content/data.csv"),
]


@dataclass
class SplitData:
    x_train: pd.DataFrame
    x_val: pd.DataFrame
    x_test: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Pipeline completo de Machine Learning para o Tech Challenge B."
    )
    parser.add_argument("--data", type=str, default=None)
    parser.add_argument("--output", type=str, default="outputs_techchallenge_b")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--validation-size", type=float, default=0.20)
    parser.add_argument("--test-size", type=float, default=0.20)
    parser.add_argument("--top-n-features", type=int, default=25)

    args, _ = parser.parse_known_args()
    return args


def resolve_dataset_path(path_arg: Optional[str]) -> Path:
    # Decide qual CSV sera usado: primeiro respeita --data; se nao vier,
    # tenta os caminhos padrao definidos acima.
    if path_arg:
        path = Path(path_arg).expanduser()
        if path.exists():
            return path
        raise FileNotFoundError(f"Dataset nao encontrado: {path}")

    for candidate in DEFAULT_DATASET_CANDIDATES:
        if candidate.exists():
            return candidate

    searched = ", ".join(str(p) for p in DEFAULT_DATASET_CANDIDATES)
    raise FileNotFoundError(
        "Dataset nao encontrado. Informe --data. Caminhos testados: " + searched
    )


def create_output_dirs(base: Path) -> Dict[str, Path]:
    # Cria a estrutura onde graficos, tabelas, modelos, relatorios e docs serao salvos.
    dirs = {
        "base": base,
        "figures": base / "figures",
        "tables": base / "tables",
        "models": base / "models",
        "reports": base / "reports",
        "docs": base / "docs",
    }
    for directory in dirs.values():
        directory.mkdir(parents=True, exist_ok=True)
    return dirs


def save_json(data: dict, path: Path) -> None:
    # Salva dicionarios em JSON legivel para facilitar auditoria dos resultados.
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)


def normalize_text(value: object) -> object:
    # Padroniza textos categoricos para reduzir duplicidade causada por caixa,
    # espacos extras e pequenos erros de digitacao.
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    text = " ".join(text.split())
    replacements = {
        "seizuers": "seizures",
        "anorexia": "loss of appetite",
        "poor appetite": "loss of appetite",
        "tiredness": "fatigue",
    }
    return replacements.get(text, text)


def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    # normaliza nomes das colunas
    df.columns = [str(col).strip() for col in df.columns]

    rename_map = {
        "Animal": "AnimalName",
        "Symptom 1": "symptoms1",
        "Symptom 2": "symptoms2",
        "Symptom 3": "symptoms3",
        "Symptom 4": "symptoms4",
        "Symptom 5": "symptoms5",
    }

    df = df.rename(columns=rename_map)

    required = {
        "AnimalName",
        "symptoms1",
        "symptoms2",
        "symptoms3",
        "symptoms4",
        "symptoms5",
        TARGET_COLUMN
    }

    missing = sorted(required.difference(df.columns))

    if missing:
        raise ValueError(f"Colunas obrigatorias ausentes no CSV: {missing}")

    return df


def clean_dataset(df_raw: pd.DataFrame) -> pd.DataFrame:
    # Executa a limpeza principal: remove duplicatas, normaliza textos,
    # converte o alvo Yes/No para 1/0 e cria features auxiliares.
    df = df_raw.copy()
    df = df.drop_duplicates().reset_index(drop=True)

    text_columns = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5"]
    for col in text_columns:
        df[col] = df[col].map(normalize_text)
        df[col] = df[col].fillna("unknown")

    target_map = {
        "yes": POSITIVE_LABEL,
        "y": POSITIVE_LABEL,
        "true": POSITIVE_LABEL,
        "1": POSITIVE_LABEL,
        "no": NEGATIVE_LABEL,
        "n": NEGATIVE_LABEL,
        "false": NEGATIVE_LABEL,
        "0": NEGATIVE_LABEL,
    }
    df[TARGET_COLUMN] = df[TARGET_COLUMN].map(normalize_text)
    df[TARGET_NAME] = df[TARGET_COLUMN].map(target_map)
    df = df.dropna(subset=[TARGET_NAME]).copy()
    df[TARGET_NAME] = df[TARGET_NAME].astype(int)

    symptom_cols = [f"symptoms{i}" for i in range(1, 6)]
    df["symptoms_text"] = df[symptom_cols].agg(" ".join, axis=1)
    df["unique_symptom_count"] = df[symptom_cols].nunique(axis=1)
    df["unknown_symptom_count"] = (df[symptom_cols] == "unknown").sum(axis=1)
    df["symptom_text_length"] = df["symptoms_text"].str.len()
    return df


def summarize_dataset(df_raw: pd.DataFrame, df: pd.DataFrame, dirs: Dict[str, Path]) -> dict:
    # Gera tabelas resumidas da base para apoiar a exploracao e alimentar o relatorio.
    symptom_cols = [f"symptoms{i}" for i in range(1, 6)]
    all_symptoms = pd.concat([df[col] for col in symptom_cols], ignore_index=True)

    summary = {
        "raw_rows": int(len(df_raw)),
        "clean_rows": int(len(df)),
        "raw_columns": list(df_raw.columns),
        "duplicates_removed": int(len(df_raw) - len(df_raw.drop_duplicates())),
        "target_distribution": df[TARGET_COLUMN].value_counts().to_dict(),
        "animal_distribution": df["AnimalName"].value_counts().to_dict(),
        "top_symptoms": all_symptoms.value_counts().head(30).to_dict(),
        "missing_values_raw": df_raw.isna().sum().to_dict(),
        "numeric_describe": df[
            ["unique_symptom_count", "unknown_symptom_count", "symptom_text_length"]
        ].describe().round(3).to_dict(),
    }
    save_json(summary, dirs["tables"] / "dataset_summary.json")

    pd.DataFrame({"missing_values": df_raw.isna().sum()}).to_csv(
        dirs["tables"] / "missing_values.csv", encoding="utf-8"
    )
    df[TARGET_COLUMN].value_counts().rename_axis("target").reset_index(name="count").to_csv(
        dirs["tables"] / "target_distribution.csv", index=False, encoding="utf-8"
    )
    all_symptoms.value_counts().rename_axis("symptom").reset_index(name="count").to_csv(
        dirs["tables"] / "symptom_frequency.csv", index=False, encoding="utf-8"
    )
    return summary


def bar_plot(series: pd.Series, title: str, xlabel: str, ylabel: str, path: Path, top_n: int = 20) -> None:
    # Funcao utilitaria para criar graficos de barras horizontais reutilizados na EDA.
    data = series.head(top_n).sort_values(ascending=True)
    height = max(4.0, min(12.0, 0.35 * len(data) + 1.5))
    plt.figure(figsize=(10, height))
    plt.barh(data.index.astype(str), data.values, color="#2F6F73")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def create_eda_plots(df_raw: pd.DataFrame, df: pd.DataFrame, dirs: Dict[str, Path]) -> List[Path]:
    # Cria graficos de EDA: alvo, animais, sintomas, features numericas e ausentes.
    figures: List[Path] = []
    symptom_cols = [f"symptoms{i}" for i in range(1, 6)]
    all_symptoms = pd.concat([df[col] for col in symptom_cols], ignore_index=True)

    path = dirs["figures"] / "01_target_distribution.png"
    bar_plot(df[TARGET_COLUMN].value_counts(), "Distribuicao do alvo", "Quantidade", "Classe", path)
    figures.append(path)

    path = dirs["figures"] / "02_top_animals.png"
    bar_plot(df["AnimalName"].value_counts(), "Animais mais frequentes", "Quantidade", "Animal", path)
    figures.append(path)

    path = dirs["figures"] / "03_top_symptoms.png"
    bar_plot(all_symptoms.value_counts(), "Sintomas mais frequentes", "Quantidade", "Sintoma", path, top_n=25)
    figures.append(path)

    path = dirs["figures"] / "04_numeric_features_by_target.png"
    numeric_cols = ["unique_symptom_count", "unknown_symptom_count", "symptom_text_length"]
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(14, 4))
    for ax, col in zip(axes, numeric_cols):
        df.boxplot(column=col, by=TARGET_COLUMN, ax=ax, grid=False)
        ax.set_title(col)
        ax.set_xlabel("Dangerous")
        ax.set_ylabel("Valor")
    fig.suptitle("Features numericas por classe")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)

    path = dirs["figures"] / "05_missing_values.png"
    missing = df_raw.isna().sum().sort_values(ascending=True)
    plt.figure(figsize=(9, 4))
    plt.barh(missing.index.astype(str), missing.values, color="#7A4E8A")
    plt.title("Valores ausentes antes da limpeza")
    plt.xlabel("Quantidade")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)
    return figures


def cramers_v(x: pd.Series, y: pd.Series) -> float:
    # Calcula Cramer's V, medida de associacao entre variaveis categoricas.
    confusion = pd.crosstab(x, y)
    if confusion.empty:
        return 0.0
    observed = confusion.to_numpy(dtype=float)
    total = observed.sum()
    if total == 0:
        return 0.0
    row_sums = observed.sum(axis=1, keepdims=True)
    col_sums = observed.sum(axis=0, keepdims=True)
    expected = row_sums @ col_sums / total
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = np.nansum((observed - expected) ** 2 / expected)
    n = total
    r, k = observed.shape
    denominator = min(k - 1, r - 1)
    if denominator <= 0:
        return 0.0
    return float(math.sqrt((chi2 / n) / denominator))


def correlation_analysis(df: pd.DataFrame, dirs: Dict[str, Path], top_n: int) -> Tuple[pd.DataFrame, List[Path]]:
    # Mede relacoes entre features e alvo com Pearson nas dummies e Cramer's V nas categoricas.
    figures: List[Path] = []
    feature_cols = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5"]
    encoded = pd.get_dummies(df[feature_cols], prefix=feature_cols)
    encoded[TARGET_NAME] = df[TARGET_NAME].values
    corr = encoded.corr(numeric_only=True)[TARGET_NAME].drop(TARGET_NAME)
    corr_table = corr.sort_values(key=lambda s: s.abs(), ascending=False).reset_index()
    corr_table.columns = ["encoded_feature", "pearson_corr_with_target"]
    corr_table.to_csv(dirs["tables"] / "encoded_feature_correlations.csv", index=False, encoding="utf-8")

    categorical_association = pd.DataFrame(
        {
            "feature": feature_cols,
            "cramers_v_with_target": [cramers_v(df[col], df[TARGET_NAME]) for col in feature_cols],
        }
    ).sort_values("cramers_v_with_target", ascending=False)
    categorical_association.to_csv(
        dirs["tables"] / "categorical_association_cramers_v.csv", index=False, encoding="utf-8"
    )

    path = dirs["figures"] / "06_top_correlations.png"
    top = corr_table.head(top_n).iloc[::-1]
    colors = np.where(top["pearson_corr_with_target"] >= 0, "#2F6F73", "#B85C5C")
    plt.figure(figsize=(11, max(5, 0.35 * len(top))))
    plt.barh(top["encoded_feature"], top["pearson_corr_with_target"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("Features codificadas mais correlacionadas com Dangerous")
    plt.xlabel("Correlacao de Pearson")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)

    path = dirs["figures"] / "07_cramers_v.png"
    top_assoc = categorical_association.iloc[::-1]
    plt.figure(figsize=(9, 4))
    plt.barh(top_assoc["feature"], top_assoc["cramers_v_with_target"], color="#4C6B9A")
    plt.title("Associacao categorica com o alvo - Cramer's V")
    plt.xlabel("Cramer's V")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)
    return corr_table, figures


def make_one_hot_encoder() -> OneHotEncoder:
    # Mantem compatibilidade com versoes novas e antigas do scikit-learn.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(categorical_cols: List[str], numeric_cols: List[str]) -> ColumnTransformer:
    # Monta o pre-processamento: one-hot para categoricas e escala para numericas.
    return ColumnTransformer(
        transformers=[
            ("categorical", make_one_hot_encoder(), categorical_cols),
            ("numeric", StandardScaler(), numeric_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def split_data(
    df: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    validation_size: float,
    test_size: float,
    random_state: int,
) -> SplitData:
    # Separa a base em treino, validacao e teste preservando a proporcao das classes.
    if validation_size <= 0 or test_size <= 0 or validation_size + test_size >= 0.8:
        raise ValueError("Use validation_size e test_size positivos, com soma menor que 0.8.")

    x = df[feature_cols]
    y = df[target_col]
    temp_size = validation_size + test_size
    x_train, x_temp, y_train, y_temp = train_test_split(
        x, y, test_size=temp_size, stratify=y, random_state=random_state
    )
    val_fraction_inside_temp = validation_size / temp_size
    x_val, x_test, y_val, y_test = train_test_split(
        x_temp,
        y_temp,
        test_size=1 - val_fraction_inside_temp,
        stratify=y_temp,
        random_state=random_state,
    )
    return SplitData(x_train, x_val, x_test, y_train, y_val, y_test)


def build_models(
    categorical_cols: List[str], numeric_cols: List[str], random_state: int
) -> Dict[str, Pipeline]:
    # Define os modelos comparados. Cada um recebe seu proprio Pipeline.
    models: Dict[str, BaseEstimator] = {
        "logistic_regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", solver="liblinear", random_state=random_state
        ),
        "decision_tree": DecisionTreeClassifier(
            max_depth=10, min_samples_leaf=3, class_weight="balanced", random_state=random_state
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=350,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "knn": KNeighborsClassifier(n_neighbors=7, weights="distance"),
    }
    return {
        name: Pipeline(
            [
                ("preprocess", build_preprocessor(categorical_cols, numeric_cols)),
                ("model", model),
            ]
        )
        for name, model in models.items()
    }


def prediction_scores(model: Pipeline, x: pd.DataFrame) -> Optional[np.ndarray]:
    # Retorna probabilidade/pontuacao da classe positiva quando o modelo permite.
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(x)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return None


def evaluate_model(model: Pipeline, x: pd.DataFrame, y: pd.Series) -> dict:
    # Calcula as metricas principais, com foco em recall e F1 da classe perigosa.
    pred = model.predict(x)
    scores = prediction_scores(model, x)
    result = {
        "accuracy": accuracy_score(y, pred),
        "precision_yes": precision_score(y, pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "recall_yes": recall_score(y, pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "f1_yes": f1_score(y, pred, pos_label=POSITIVE_LABEL, zero_division=0),
        "f1_weighted": f1_score(y, pred, average="weighted", zero_division=0),
    }
    if scores is not None and len(np.unique(y)) == 2:
        result["roc_auc"] = roc_auc_score(y, scores)
    else:
        result["roc_auc"] = np.nan
    return result


def train_and_select_models(
    split: SplitData,
    categorical_cols: List[str],
    numeric_cols: List[str],
    random_state: int,
    dirs: Dict[str, Path],
) -> Tuple[str, Pipeline, pd.DataFrame]:
    # Treina todos os modelos, avalia na validacao e escolhe o melhor por recall/F1.
    models = build_models(categorical_cols, numeric_cols, random_state)

    records = []
    for name, model in models.items():
        print(f"Treinando modelo: {name}")
        model.fit(split.x_train, split.y_train)
        val_metrics = evaluate_model(model, split.x_val, split.y_val)
        records.append({"model": name, **val_metrics})

    validation_table = pd.DataFrame(records).sort_values(
        ["recall_yes", "f1_yes", "accuracy"], ascending=False
    )
    validation_table.to_csv(dirs["tables"] / "validation_metrics.csv", index=False, encoding="utf-8")
    best_name = str(validation_table.iloc[0]["model"])
    best_model = models[best_name]
    return best_name, best_model, validation_table


def save_confusion_matrix(y_true: pd.Series, y_pred: np.ndarray, title: str, path: Path) -> None:
    # Gera matriz de confusao para visualizar acertos e erros entre No e Yes.
    matrix = confusion_matrix(y_true, y_pred, labels=[NEGATIVE_LABEL, POSITIVE_LABEL])
    display = ConfusionMatrixDisplay(
        confusion_matrix=matrix, display_labels=["No", "Yes"]
    )
    display.plot(cmap="Blues", values_format="d")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def evaluate_best_model(
    model_name: str,
    model: Pipeline,
    split: SplitData,
    dirs: Dict[str, Path],
) -> Tuple[pd.DataFrame, dict, List[Path]]:
    # Avalia o modelo escolhido em treino, validacao e teste, salvando metricas e graficos.
    figures: List[Path] = []
    datasets = {
        "train": (split.x_train, split.y_train),
        "validation": (split.x_val, split.y_val),
        "test": (split.x_test, split.y_test),
    }

    records = []
    reports = {}
    for dataset_name, (x_part, y_part) in datasets.items():
        pred = model.predict(x_part)
        metrics = evaluate_model(model, x_part, y_part)
        records.append({"dataset": dataset_name, "model": model_name, **metrics})
        reports[dataset_name] = classification_report(
            y_part,
            pred,
            target_names=["No - nao perigoso", "Yes - perigoso"],
            output_dict=True,
            zero_division=0,
        )
        path = dirs["figures"] / f"08_confusion_matrix_{dataset_name}.png"
        save_confusion_matrix(y_part, pred, f"Matriz de confusao - {dataset_name}", path)
        figures.append(path)

    metrics_table = pd.DataFrame(records)
    metrics_table.to_csv(dirs["tables"] / "best_model_metrics.csv", index=False, encoding="utf-8")
    save_json(reports, dirs["tables"] / "classification_reports.json")
    return metrics_table, reports, figures


def get_feature_names(model: Pipeline, original_cols: List[str]) -> np.ndarray:
    # Recupera os nomes finais das features apos o pre-processamento.
    preprocessor = model.named_steps["preprocess"]
    try:
        return preprocessor.get_feature_names_out(original_cols)
    except Exception:
        names: List[str] = []
        categorical_encoder = preprocessor.named_transformers_.get("categorical")
        if categorical_encoder is not None and hasattr(categorical_encoder, "get_feature_names_out"):
            cat_cols = preprocessor.transformers_[0][2]
            names.extend(categorical_encoder.get_feature_names_out(cat_cols).tolist())
        numeric_cols = preprocessor.transformers_[1][2]
        names.extend([f"numeric__{col}" for col in numeric_cols])
        return np.array(names)


def model_feature_importance(
    model: Pipeline,
    feature_cols: List[str],
    split: SplitData,
    dirs: Dict[str, Path],
    top_n: int,
) -> Tuple[pd.DataFrame, List[Path]]:
    # Interpreta o modelo usando importancia nativa, coeficientes ou permutation importance.
    figures: List[Path] = []
    estimator = model.named_steps["model"]
    feature_names = get_feature_names(model, feature_cols)

    if hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        method = "model_feature_importances"
    elif hasattr(estimator, "coef_"):
        values = np.ravel(np.abs(estimator.coef_))
        method = "absolute_model_coefficients"
    else:
        print("Modelo sem importance nativa. Calculando permutation importance no teste.")
        permutation = permutation_importance(
            model,
            split.x_test,
            split.y_test,
            n_repeats=12,
            random_state=42,
            scoring="f1",
            n_jobs=-1,
        )
        feature_names = np.array(feature_cols)
        values = permutation.importances_mean
        method = "permutation_importance"

    length = min(len(feature_names), len(values))
    table = pd.DataFrame(
        {
            "feature": feature_names[:length],
            "importance": values[:length],
            "method": method,
        }
    ).sort_values("importance", ascending=False)
    table.to_csv(dirs["tables"] / "feature_importance.csv", index=False, encoding="utf-8")

    path = dirs["figures"] / "09_feature_importance.png"
    top = table.head(top_n).iloc[::-1]
    plt.figure(figsize=(11, max(5, 0.35 * len(top))))
    plt.barh(top["feature"], top["importance"], color="#5A7D3A")
    plt.title(f"Top {len(top)} features - {method}")
    plt.xlabel("Importancia")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()
    figures.append(path)
    return table, figures


def try_generate_shap(
    model: Pipeline,
    split: SplitData,
    feature_cols: List[str],
    dirs: Dict[str, Path],
    top_n: int,
) -> Tuple[Optional[Path], str]:
    # Tenta gerar explicabilidade com SHAP; se falhar, o pipeline continua normalmente.
    try:
        import shap  # type: ignore
    except Exception:
        return None, "SHAP nao instalado. Foi usado feature importance/permutation importance como fallback."

    try:
        transformed = model.named_steps["preprocess"].transform(split.x_test)
        feature_names = get_feature_names(model, feature_cols)
        if hasattr(transformed, "toarray"):
            transformed = transformed.toarray()
        sample_size = min(200, transformed.shape[0])
        x_sample = transformed[:sample_size]
        estimator = model.named_steps["model"]

        explainer = shap.Explainer(estimator, x_sample, feature_names=feature_names)
        shap_values = explainer(x_sample)

        plt.figure()
        shap.plots.bar(shap_values, max_display=top_n, show=False)
        path = dirs["figures"] / "10_shap_summary_bar.png"
        plt.tight_layout()
        plt.savefig(path, dpi=160, bbox_inches="tight")
        plt.close()
        return path, "SHAP executado com sucesso para uma amostra do conjunto de teste."
    except Exception as exc:
        message = (
            "SHAP estava instalado, mas falhou neste modelo/ambiente. "
            f"Fallback mantido com feature importance. Erro: {exc}"
        )
        return None, message


def write_prediction_example(model: Pipeline, split: SplitData, dirs: Dict[str, Path]) -> dict:
    # Salva um exemplo de predicao para demonstrar o uso do modelo em triagem.
    sample = split.x_test.iloc[[0]].copy()
    pred = int(model.predict(sample)[0])
    scores = prediction_scores(model, sample)
    probability = float(scores[0]) if scores is not None else None
    result = {
        "input": sample.iloc[0].to_dict(),
        "prediction_binary": pred,
        "prediction_label": "Yes - perigoso" if pred == POSITIVE_LABEL else "No - nao perigoso",
        "probability_yes": probability,
        "warning": "A previsao e apenas apoio a triagem. A decisao final deve ser do profissional de saude.",
    }
    save_json(result, dirs["tables"] / "prediction_example.json")
    return result


def write_readme_and_dockerfile(dirs: Dict[str, Path]) -> Tuple[Path, Path, Path]:
    # Gera README, Dockerfile e roteiro sugerido para o video de demonstracao.
    readme = dirs["docs"] / "README.md"
    dockerfile = dirs["docs"] / "Dockerfile"
    video_script = dirs["docs"] / "roteiro_video_demo.md"

    readme.write_text(
        textwrap.dedent(
            """
            # Tech Challenge B - Pipeline de IA

            Este projeto executa um pipeline completo de Machine Learning para classificar
            se um caso deve ser marcado como perigoso a partir de animal e sintomas.

            ## Execucao local

            ```bash
            python techchallengeB.py --data data.csv --output outputs_techchallenge_b
            ```

            ## Principais artefatos gerados

            - `reports/relatorio_tecnico.md`
            - `reports/relatorio_tecnico.pdf`
            - `tables/*.csv` e `tables/*.json`
            - `figures/*.png`
            - `models/best_model.pkl`

            ## Docker

            Copie `techchallengeB.py`, `data.csv` e este Dockerfile para a mesma pasta:

            ```bash
            docker build -t techchallenge-b .
            docker run --rm -v "%cd%/outputs:/app/outputs" techchallenge-b
            ```
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )

    dockerfile.write_text(
        textwrap.dedent(
            """
            FROM python:3.11-slim

            WORKDIR /app
            COPY techchallengeB.py /app/techchallengeB.py
            COPY data.csv /app/data.csv

            RUN pip install --no-cache-dir pandas numpy matplotlib scikit-learn shap

            CMD ["python", "techchallengeB.py", "--data", "/app/data.csv", "--output", "/app/outputs"]
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )

    video_script.write_text(
        textwrap.dedent(
            """
            # Roteiro sugerido para video de demonstracao

            1. Mostrar a estrutura do projeto e o arquivo unico `techchallengeB.py`.
            2. Executar o script com `python techchallengeB.py --data data.csv`.
            3. Abrir a pasta `outputs_techchallenge_b`.
            4. Apresentar graficos de EDA, correlacao e matriz de confusao.
            5. Mostrar as metricas de validacao/teste e explicar a escolha do melhor modelo.
            6. Mostrar feature importance/SHAP e explicar limitacoes.
            7. Reforcar que o sistema e apoio a triagem e que o profissional tem a decisao final.
            """
        ).strip()
        + "\n",
        encoding="utf-8",
    )
    return readme, dockerfile, video_script


def dataframe_to_markdown(df: pd.DataFrame) -> str:
    # Converte DataFrames em Markdown sem depender do pacote opcional tabulate.
    text_df = df.copy()
    for col in text_df.columns:
        if pd.api.types.is_float_dtype(text_df[col]):
            text_df[col] = text_df[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
        else:
            text_df[col] = text_df[col].map(lambda value: "" if pd.isna(value) else str(value))

    headers = [str(col) for col in text_df.columns]
    rows = text_df.values.tolist()
    widths = [
        max(len(headers[index]), *(len(str(row[index])) for row in rows)) if rows else len(headers[index])
        for index in range(len(headers))
    ]

    def render_row(values: Iterable[object]) -> str:
        return "| " + " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(values)) + " |"

    separator = "| " + " | ".join("-" * width for width in widths) + " |"
    lines = [render_row(headers), separator]
    lines.extend(render_row(row) for row in rows)
    return "\n".join(lines)


def format_metrics_table(metrics: pd.DataFrame) -> str:
    # Seleciona e formata as metricas mais relevantes para o relatorio.
    cols = ["dataset", "accuracy", "precision_yes", "recall_yes", "f1_yes", "f1_weighted", "roc_auc"]
    available_cols = [col for col in cols if col in metrics.columns]
    return dataframe_to_markdown(metrics[available_cols].round(4))


def write_markdown_report(
    dataset_path: Path,
    summary: dict,
    validation_metrics: pd.DataFrame,
    best_model_name: str,
    best_metrics: pd.DataFrame,
    feature_importance: pd.DataFrame,
    shap_message: str,
    figures: Iterable[Path],
    docs: Tuple[Path, Path, Path],
    dirs: Dict[str, Path],
) -> Path:
    # Escreve o relatorio tecnico em Markdown com metodologia, metricas e limitacoes.
    top_features = feature_importance.head(15)[["feature", "importance"]].copy()
    top_features["importance"] = top_features["importance"].round(5)
    validation_text = dataframe_to_markdown(validation_metrics.round(4))
    best_text = format_metrics_table(best_metrics)
    features_text = dataframe_to_markdown(top_features)
    figure_lines = "\n".join(f"- `{path}`" for path in figures)
    readme, dockerfile, video_script = docs

    report = f"""
# Relatorio tecnico - Tech Challenge B

## Problema escolhido

O dataset analisado e o arquivo anexado `data.csv`, localizado em `{dataset_path}`.
A tarefa foi formulada como um problema de classificacao binaria: prever se um
caso deve ser marcado como `Dangerous = Yes` ou `Dangerous = No` a partir do
animal e de cinco sintomas observados.

Embora o enunciado use exemplos de diagnostico humano, este dataset representa
um cenario clinico/veterinario de triagem. A solucao deve ser interpretada como
apoio inicial a decisao, nunca como diagnostico final automatico.

## Exploracao dos dados

- Linhas originais: {summary["raw_rows"]}
- Linhas apos limpeza: {summary["clean_rows"]}
- Duplicatas removidas: {summary["duplicates_removed"]}
- Colunas: {", ".join(summary["raw_columns"])}
- Distribuicao do alvo: {summary["target_distribution"]}

Foram gerados graficos de distribuicao do alvo, animais mais frequentes,
sintomas mais frequentes, features numericas derivadas e valores ausentes.

## Pre-processamento

As estrategias utilizadas foram:

- padronizacao de texto com `strip`, caixa baixa e normalizacao de espacos;
- correcao de algumas inconsistencias textuais evidentes, como `seizuers` para `seizures`;
- remocao de duplicatas;
- mapeamento de `Dangerous` para alvo binario;
- preenchimento de sintomas ausentes com `unknown`;
- criacao de features numericas derivadas: quantidade de sintomas unicos,
  quantidade de sintomas desconhecidos e tamanho textual dos sintomas;
- `OneHotEncoder` para variaveis categoricas;
- `StandardScaler` para features numericas;
- pipeline do scikit-learn para evitar vazamento de dados entre treino, validacao e teste.

## Correlacao

A correlacao foi calculada de duas formas:

- correlacao de Pearson entre dummies one-hot e o alvo binario;
- Cramer's V entre cada variavel categorica original e o alvo.

Essas tabelas foram salvas em `tables/encoded_feature_correlations.csv` e
`tables/categorical_association_cramers_v.csv`.

## Modelagem

Foram treinados quatro modelos, cobrindo tecnicas lineares, arvores e metodos
baseados em vizinhanca:

- Regressao Logistica;
- Arvore de Decisao;
- Random Forest;
- KNN.

A separacao foi feita em treino, validacao e teste. O conjunto de validacao foi
usado para escolher o melhor modelo, priorizando `recall` da classe `Yes`, pois
em triagem clinica e mais grave deixar de sinalizar um caso perigoso do que
gerar um alerta falso. O F1-score foi usado como criterio de equilibrio.

### Metricas de validacao

{validation_text}

Modelo selecionado: `{best_model_name}`.

### Metricas do modelo selecionado

{best_text}

## Interpretacao

O script gera importancia de variaveis nativa quando o modelo permite
(`feature_importances_` ou coeficientes). Quando isso nao esta disponivel,
usa permutation importance. SHAP e executado automaticamente quando a biblioteca
esta instalada e compativel com o modelo selecionado.

Status SHAP: {shap_message}

Top features:

{features_text}

## Analise critica

O modelo pode ser util como ferramenta de apoio a triagem, destacando casos que
merecem revisao prioritaria. Entretanto, seu uso pratico exige cuidado:

- o dataset e pequeno e limitado ao universo de animais/sintomas observados;
- categorias novas podem aparecer em producao;
- correlacao nao implica causalidade;
- sintomas textuais foram tratados como categorias, sem contexto clinico amplo;
- e necessario validar o modelo com dados externos e acompanhamento de
  profissionais antes de qualquer uso real.

Assim, a recomendacao e usar a solucao como alerta inicial e ferramenta de
organizacao do fluxo de atendimento. A decisao final deve permanecer com o(a)
medico(a) ou profissional responsavel.

## Artefatos gerados

- Modelo: `models/best_model.pkl`
- Metricas: `tables/validation_metrics.csv` e `tables/best_model_metrics.csv`
- Relatorios de classificacao: `tables/classification_reports.json`
- Exemplo de predicao: `tables/prediction_example.json`
- README: `{readme}`
- Dockerfile: `{dockerfile}`
- Roteiro do video: `{video_script}`

## Figuras

{figure_lines}
"""
    path = dirs["reports"] / "relatorio_tecnico.md"
    path.write_text(textwrap.dedent(report).strip() + "\n", encoding="utf-8")
    return path


def add_text_page(pdf: PdfPages, title: str, body: str) -> None:
    # Adiciona uma pagina textual ao PDF gerado pelo script.
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor("white")
    plt.axis("off")
    wrapped = "\n".join(textwrap.wrap(body, width=92))
    fig.text(0.08, 0.94, title, fontsize=16, weight="bold", va="top")
    fig.text(0.08, 0.88, wrapped, fontsize=10, va="top", linespacing=1.35)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def add_image_page(pdf: PdfPages, image_path: Path, title: str) -> None:
    # Adiciona uma imagem/grafico como pagina no PDF do relatorio.
    if not image_path.exists():
        return
    image = plt.imread(image_path)
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.patch.set_facecolor("white")
    plt.axis("off")
    fig.text(0.08, 0.96, title, fontsize=14, weight="bold", va="top")
    ax = fig.add_axes([0.08, 0.08, 0.84, 0.82])
    ax.imshow(image)
    ax.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def write_pdf_report(
    markdown_report: Path,
    figures: Iterable[Path],
    best_metrics: pd.DataFrame,
    dirs: Dict[str, Path],
) -> Path:
    # Converte o relatorio e os principais graficos em um PDF simples.
    path = dirs["reports"] / "relatorio_tecnico.pdf"
    text = markdown_report.read_text(encoding="utf-8")
    plain = text.replace("#", "").replace("`", "")
    chunks = textwrap.wrap(plain, width=4800, replace_whitespace=False)

    with PdfPages(path) as pdf:
        for index, chunk in enumerate(chunks[:4], start=1):
            add_text_page(pdf, f"Relatorio tecnico - parte {index}", chunk)

        metrics_body = best_metrics.round(4).to_string(index=False)
        add_text_page(pdf, "Metricas do modelo selecionado", metrics_body)

        for fig_path in figures:
            add_image_page(pdf, fig_path, fig_path.stem.replace("_", " ").title())

    return path


def save_model(model: Pipeline, dirs: Dict[str, Path], metadata: dict) -> Tuple[Path, Path]:
    # Salva o modelo treinado com pickle e grava metadados sobre a selecao.
    model_path = dirs["models"] / "best_model.pkl"
    metadata_path = dirs["models"] / "model_metadata.json"
    save_json(metadata, metadata_path)

    with model_path.open("wb") as file:
        pickle.dump(model, file)
    return model_path, metadata_path


def main() -> None:
    # Orquestra o pipeline completo: entrada, limpeza, EDA, treino, avaliacao e exportacao.
    args = parse_args()
    dataset_path = resolve_dataset_path(args.data)
    dirs = create_output_dirs(Path(args.output))

    print("Carregando dataset:", dataset_path)
    df_raw = load_dataset(dataset_path)
    df = clean_dataset(df_raw)

    summary = summarize_dataset(df_raw, df, dirs)
    figures = create_eda_plots(df_raw, df, dirs)
    _, corr_figures = correlation_analysis(df, dirs, args.top_n_features)
    figures.extend(corr_figures)

    categorical_cols = ["AnimalName", "symptoms1", "symptoms2", "symptoms3", "symptoms4", "symptoms5"]
    numeric_cols = ["unique_symptom_count", "unknown_symptom_count", "symptom_text_length"]
    feature_cols = categorical_cols + numeric_cols

    split = split_data(
        df,
        feature_cols,
        TARGET_NAME,
        validation_size=args.validation_size,
        test_size=args.test_size,
        random_state=args.random_state,
    )

    split_summary = {
        "train_rows": int(len(split.x_train)),
        "validation_rows": int(len(split.x_val)),
        "test_rows": int(len(split.x_test)),
        "feature_columns": feature_cols,
        "categorical_columns": categorical_cols,
        "numeric_columns": numeric_cols,
    }
    save_json(split_summary, dirs["tables"] / "split_summary.json")

    best_name, best_model, validation_metrics = train_and_select_models(
        split, categorical_cols, numeric_cols, args.random_state, dirs
    )

    best_metrics, reports, confusion_figures = evaluate_best_model(best_name, best_model, split, dirs)
    figures.extend(confusion_figures)

    importance, importance_figures = model_feature_importance(
        best_model, feature_cols, split, dirs, args.top_n_features
    )
    figures.extend(importance_figures)

    shap_path, shap_message = try_generate_shap(
        best_model, split, feature_cols, dirs, args.top_n_features
    )
    if shap_path is not None:
        figures.append(shap_path)

    prediction_example = write_prediction_example(best_model, split, dirs)
    docs = write_readme_and_dockerfile(dirs)

    metadata = {
        "best_model": best_name,
        "dataset_path": str(dataset_path),
        "target": TARGET_NAME,
        "positive_label": "Dangerous=Yes",
        "selection_rule": "Maior recall_yes na validacao; desempate por f1_yes e accuracy.",
        "test_metrics": best_metrics[best_metrics["dataset"] == "test"].to_dict(orient="records"),
        "shap_status": shap_message,
    }
    model_path, metadata_path = save_model(best_model, dirs, metadata)

    markdown_report = write_markdown_report(
        dataset_path=dataset_path,
        summary=summary,
        validation_metrics=validation_metrics,
        best_model_name=best_name,
        best_metrics=best_metrics,
        feature_importance=importance,
        shap_message=shap_message,
        figures=figures,
        docs=docs,
        dirs=dirs,
    )
    pdf_report = write_pdf_report(markdown_report, figures, best_metrics, dirs)

    print("\nPipeline concluido.")
    print(f"Modelo selecionado: {best_name}")
    print("\nMetricas do modelo selecionado:")
    print(best_metrics.round(4).to_string(index=False))
    print("\nExemplo de predicao:")
    print(json.dumps(prediction_example, indent=2, ensure_ascii=False))
    print("\nArtefatos principais:")
    print(f"- Modelo: {model_path}")
    print(f"- Metadados: {metadata_path}")
    print(f"- Relatorio Markdown: {markdown_report}")
    print(f"- Relatorio PDF: {pdf_report}")
    print(f"- README/Dockerfile/Roteiro: {docs[0].parent}")


if __name__ == "__main__":
    main()


Carregando dataset: data.csv
Treinando modelo: logistic_regression
Treinando modelo: decision_tree
Treinando modelo: random_forest
Treinando modelo: knn



Pipeline concluido.
Modelo selecionado: random_forest

Metricas do modelo selecionado:
   dataset         model  accuracy  precision_yes  recall_yes  f1_yes  f1_weighted  roc_auc
     train random_forest    1.0000          1.000         1.0  1.0000       1.0000   1.0000
validation random_forest    0.9821          0.982         1.0  0.9909       0.9769   0.9070
      test random_forest    0.9882          0.988         1.0  0.9940       0.9862   0.9667

Exemplo de predicao:
{
  "input": {
    "AnimalName": "buffaloes",
    "symptoms1": "fever",
    "symptoms2": "dyspnea",
    "symptoms3": "coughing",
    "symptoms4": "lethargy",
    "symptoms5": "loss of appetite",
    "unique_symptom_count": 5,
    "unknown_symptom_count": 0,
    "symptom_text_length": 48
  },
  "prediction_binary": 1,
  "prediction_label": "Yes - perigoso",
  "probability_yes": 0.8087676777370568,
  "warning": "A previsao e apenas apoio a triagem. A decisao final deve ser do profissional de saude."
}

Artefatos princi

<Figure size 640x480 with 0 Axes>

IMPLEMENTAÇÃO DO ALGORITMO GENÉTICO DA FASE 2

In [ ]:
# ============================================================
# MÓDULO 2 - OTIMIZAÇÃO DE HIPERPARÂMETROS COM ALGORITMO GENÉTICO
# Baseado integralmente no pipeline desenvolvido no Módulo 1
# ============================================================

from copy import deepcopy
import json
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate


# ------------------------------------------------------------
# 1. FITNESS
# ------------------------------------------------------------
# Como o problema é de triagem de casos perigosos, damos maior
# peso ao RECALL da classe perigosa.
#
# Fitness =
# 30% Accuracy
# 40% Recall
# 30% F1-score
# ------------------------------------------------------------

FITNESS_WEIGHTS = {
    "accuracy": 0.30,
    "recall": 0.40,
    "f1": 0.30,
}


# ------------------------------------------------------------
# 2. ESPAÇO DE BUSCA DOS HIPERPARÂMETROS
# ------------------------------------------------------------
# Cada entrada representa um gene do indivíduo.
#
# Valores discretos -> lista
# Intervalos inteiros -> tupla (mínimo, máximo)
# ------------------------------------------------------------

GENETIC_SPACES = {

    "logistic_regression": {

        "C": [
            0.01,
            0.05,
            0.1,
            0.5,
            1.0,
            2.0,
            5.0,
            10.0
        ],

        "penalty": [
            "l1",
            "l2"
        ],

        "class_weight": [
            "balanced",
            None
        ],
    },

    "decision_tree": {

        "max_depth": (
            3,
            30
        ),

        "min_samples_split": (
            2,
            20
        ),

        "min_samples_leaf": (
            1,
            10
        ),

        "max_features": [
            "sqrt",
            "log2",
            None
        ],

        "class_weight": [
            "balanced",
            None
        ],
    },

    "random_forest": {

        "n_estimators": (
            150,
            600
        ),

        "max_depth": (
            4,
            26
        ),

        "min_samples_split": (
            2,
            16
        ),

        "min_samples_leaf": (
            1,
            9
        ),

        "max_features": [
            "sqrt",
            "log2",
            None
        ],

        "class_weight": [
            "balanced",
            "balanced_subsample",
            None
        ],
    },

    "knn": {

        "n_neighbors": (
            3,
            25
        ),

        "weights": [
            "uniform",
            "distance"
        ],

        "p": (
            1,
            2
        ),

        "leaf_size": (
            15,
            60
        ),
    },
}


# ------------------------------------------------------------
# 3. CONSTRUÇÃO DO MODELO
# ------------------------------------------------------------
# IMPORTANTE:
#
# Não criamos um novo pipeline.
#
# Reutilizamos build_models() da Célula 1.
#
# O algoritmo genético somente altera os hiperparâmetros
# do modelo localizado dentro do Pipeline.
# ------------------------------------------------------------

def build_genetic_model(
    model_name,
    genes,
    categorical_cols,
    numeric_cols,
    random_state
):

    models = build_models(
        categorical_cols,
        numeric_cols,
        random_state
    )

    model = models[model_name]

    # Atualiza somente os hiperparâmetros do estimador final.
    model.set_params(
        **{
            f"model__{parameter}": value
            for parameter, value in genes.items()
        }
    )

    return model


# ------------------------------------------------------------
# 4. CRIAÇÃO DE UM INDIVÍDUO
# ------------------------------------------------------------

def create_individual(search_space, rng):

    individual = {}

    for gene, values in search_space.items():

        if isinstance(values, tuple):

            value = int(
                rng.integers(
                    values[0],
                    values[1] + 1
                )
            )

        else:

            value = values[
                int(
                    rng.integers(
                        0,
                        len(values)
                    )
                )
            ]

        individual[gene] = value

    return individual


# ------------------------------------------------------------
# 5. AVALIAÇÃO DO INDIVÍDUO
# ------------------------------------------------------------

def evaluate_individual(
    model_name,
    individual,
    x_train,
    y_train,
    categorical_cols,
    numeric_cols,
    random_state
):

    model = build_genetic_model(
        model_name,
        individual,
        categorical_cols,
        numeric_cols,
        random_state
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    scoring = {
        "accuracy": "accuracy",
        "recall": "recall",
        "f1": "f1",
    }

    scores = cross_validate(
        model,
        x_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    accuracy = float(
        np.mean(scores["test_accuracy"])
    )

    recall = float(
        np.mean(scores["test_recall"])
    )

    f1 = float(
        np.mean(scores["test_f1"])
    )

    fitness = (
        FITNESS_WEIGHTS["accuracy"] * accuracy
        + FITNESS_WEIGHTS["recall"] * recall
        + FITNESS_WEIGHTS["f1"] * f1
    )

    return {
        "accuracy": accuracy,
        "recall": recall,
        "f1": f1,
        "fitness": fitness,
    }


# ------------------------------------------------------------
# 6. MUTAÇÃO
# ------------------------------------------------------------

def mutate(
    individual,
    search_space,
    rng
):

    child = individual.copy()

    gene = str(
        rng.choice(
            list(search_space.keys())
        )
    )

    values = search_space[gene]

    if isinstance(values, tuple):

        child[gene] = int(
            rng.integers(
                values[0],
                values[1] + 1
            )
        )

    else:

        child[gene] = values[
            int(
                rng.integers(
                    0,
                    len(values)
                )
            )
        ]

    return child


# ------------------------------------------------------------
# 7. CRUZAMENTO
# ------------------------------------------------------------

def crossover(
    parent1,
    parent2,
    search_space,
    rng
):

    child = {}

    for gene in search_space:

        if rng.random() < 0.5:

            child[gene] = parent1[gene]

        else:

            child[gene] = parent2[gene]

    return child


# ------------------------------------------------------------
# 8. BUSCA GENÉTICA
# ------------------------------------------------------------

def genetic_search_model(
    model_name,
    x_train,
    y_train,
    categorical_cols,
    numeric_cols,
    random_state=42,
    population_size=8,
    generations=5,
    mutation_rate=0.30
):

    rng = np.random.default_rng(
        random_state
    )

    search_space = GENETIC_SPACES[
        model_name
    ]

    # Criação da população inicial
    population = [
        create_individual(
            search_space,
            rng
        )
        for _ in range(population_size)
    ]

    history = []

    best_individual = None
    best_metrics = None
    best_fitness = -np.inf

    # --------------------------------------------------------
    # EVOLUÇÃO DAS GERAÇÕES
    # --------------------------------------------------------

    for generation in range(generations):

        scored_population = []

        for individual in population:

            metrics = evaluate_individual(
                model_name=model_name,
                individual=individual,
                x_train=x_train,
                y_train=y_train,
                categorical_cols=categorical_cols,
                numeric_cols=numeric_cols,
                random_state=random_state
            )

            scored_population.append(
                (
                    individual,
                    metrics
                )
            )

        # Ordena pela maior fitness
        scored_population.sort(
            key=lambda item: item[1]["fitness"],
            reverse=True
        )

        generation_best_individual = (
            scored_population[0][0]
        )

        generation_best_metrics = (
            scored_population[0][1]
        )

        # Guarda histórico
        history.append(
            {
                "model": model_name,
                "generation": generation + 1,
                "fitness": generation_best_metrics["fitness"],
                "accuracy_cv": generation_best_metrics["accuracy"],
                "recall_cv": generation_best_metrics["recall"],
                "f1_cv": generation_best_metrics["f1"],
            }
        )

        # Atualiza melhor indivíduo global
        if (
            generation_best_metrics["fitness"]
            > best_fitness
        ):

            best_individual = deepcopy(
                generation_best_individual
            )

            best_metrics = deepcopy(
                generation_best_metrics
            )

            best_fitness = (
                generation_best_metrics["fitness"]
            )

        # ----------------------------------------------------
        # SELEÇÃO
        # ----------------------------------------------------
        #
        # Elitismo:
        # mantém os melhores indivíduos.
        # ----------------------------------------------------

        elite_count = max(
            2,
            population_size // 4
        )

        parents = [
            individual
            for individual, _ in scored_population[
                :elite_count
            ]
        ]

        # Próxima população começa com os melhores
        next_population = [
            parent.copy()
            for parent in parents
        ]

        # ----------------------------------------------------
        # CRUZAMENTO + MUTAÇÃO
        # ----------------------------------------------------

        while len(next_population) < population_size:

            parent1 = parents[
                int(
                    rng.integers(
                        0,
                        len(parents)
                    )
                )
            ]

            parent2 = parents[
                int(
                    rng.integers(
                        0,
                        len(parents)
                    )
                )
            ]

            child = crossover(
                parent1,
                parent2,
                search_space,
                rng
            )

            # Mutação
            if rng.random() < mutation_rate:

                child = mutate(
                    child,
                    search_space,
                    rng
                )

            next_population.append(
                child
            )

        population = next_population

        print(
            f"{model_name} | "
            f"Geração {generation + 1}/{generations} | "
            f"Fitness: {generation_best_metrics['fitness']:.4f} | "
            f"Recall: {generation_best_metrics['recall']:.4f} | "
            f"F1: {generation_best_metrics['f1']:.4f}"
        )

    # --------------------------------------------------------
    # TREINA O MELHOR MODELO ENCONTRADO
    # --------------------------------------------------------

    best_model = build_genetic_model(
        model_name=model_name,
        genes=best_individual,
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        random_state=random_state
    )

    best_model.fit(
        x_train,
        y_train
    )

    return (
        best_model,
        best_individual,
        best_metrics,
        history
    )


# ------------------------------------------------------------
# 9. TRÊS EXPERIMENTOS OBRIGATÓRIOS
# ------------------------------------------------------------
#
# Experimento 1:
# população pequena + baixa mutação
#
# Experimento 2:
# população média + mutação média
#
# Experimento 3:
# população maior + mutação alta
# ------------------------------------------------------------

GENETIC_EXPERIMENTS = [

    {
        "experiment": "Experimento 1",
        "population_size": 6,
        "generations": 4,
        "mutation_rate": 0.10,
    },

    {
        "experiment": "Experimento 2",
        "population_size": 8,
        "generations": 5,
        "mutation_rate": 0.30,
    },

    {
        "experiment": "Experimento 3",
        "population_size": 12,
        "generations": 6,
        "mutation_rate": 0.50,
    },
]


Execução e comparação com o Modelo tabular da Fase 1 e Algoritmo Genético da Fase 2

In [ ]:
"""============================================================
CÉLULA 3 - EXECUÇÃO INTEGRADA DO PIPELINE + ALGORITMO GENÉTICO
============================================================
MÓDULO 3 - EXECUÇÃO INTEGRADA
Esta célula depende das Células 1 e 2.
Célula 1:
- carrega e limpa os dados;
- cria as funções do pipeline;
- cria train/validation/test;
- possui os modelos originais;
- possui avaliação, importance, SHAP e relatórios.
Célula 2:
- possui GENETIC_SPACES;
- possui GENETIC_EXPERIMENTS;
- possui genetic_search_model();
- possui build_genetic_model();
- possui avaliação genética.
Esta célula apenas ORQUESTRA as duas anteriores.
============================================================
"""

from pathlib import Path
from copy import deepcopy
import json
import pickle

import numpy as np
import pandas as pd

"""============================================================
1. CONFIGURAÇÃO
============================================================
"""

RANDOM_STATE = 42

# Dataset utilizado pela Célula 1.

DATASET_PATH = Path("/content/data.csv")

# Pasta exclusiva para os resultados da otimização genética.

OUTPUT_DIR = Path("/content/outputs_techchallenge_genetico")

# Número de features mostradas nos gráficos.

TOP_N_FEATURES = 25

"""============================================================
2. CRIAÇÃO DAS PASTAS
============================================================
"""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENETIC_TABLES_DIR = OUTPUT_DIR / "tables"
GENETIC_MODELS_DIR = OUTPUT_DIR / "models"
GENETIC_FIGURES_DIR = OUTPUT_DIR / "figures"

GENETIC_TABLES_DIR.mkdir(parents=True, exist_ok=True)
GENETIC_MODELS_DIR.mkdir(parents=True, exist_ok=True)
GENETIC_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

"""============================================================
3. VALIDAR DEPENDÊNCIAS DAS CÉLULAS 1 E 2
============================================================
"""

required_cell_1 = [
    "load_dataset",
    "clean_dataset",
    "split_data",
    "build_models",
    "evaluate_best_model",
    "model_feature_importance",
    "try_generate_shap",
    "write_prediction_example",
]

required_cell_2 = [
    "GENETIC_SPACES",
    "GENETIC_EXPERIMENTS",
    "genetic_search_model",
]

missing_cell_1 = [
    name for name in required_cell_1
    if name not in globals()
]

missing_cell_2 = [
    name for name in required_cell_2
    if name not in globals()
]

if missing_cell_1:
    raise RuntimeError(
        "A Célula 1 precisa ser executada antes da Célula 3.\n"
        f"Funções ausentes: {missing_cell_1}"
    )

if missing_cell_2:
    raise RuntimeError(
        "A Célula 2 precisa ser executada antes da Célula 3.\n"
        f"Objetos/funções ausentes: {missing_cell_2}"
    )

print("✓ Dependências da Célula 1 encontradas.")
print("✓ Dependências da Célula 2 encontradas.")

"""============================================================
4. CARREGAMENTO E LIMPEZA
============================================================
"""

print("\n" + "=" * 70)
print("1. CARREGAMENTO E LIMPEZA DOS DADOS")
print("=" * 70)

if not DATASET_PATH.exists():
    # Aproveita a lógica de busca da Célula 1 caso o arquivo
    # não esteja no caminho informado.
    DATASET_PATH = resolve_dataset_path(None)

print(f"Dataset: {DATASET_PATH}")

df_raw = load_dataset(DATASET_PATH)
df = clean_dataset(df_raw)

print(f"Linhas originais: {len(df_raw)}")
print(f"Linhas após limpeza: {len(df)}")
print(f"Duplicatas removidas: {len(df_raw) - len(df_raw.drop_duplicates())}")
print(f"Distribuição do alvo:")
print(df[TARGET_NAME].value_counts())

"""============================================================
5. DEFINIÇÃO DAS FEATURES
Utiliza EXATAMENTE as mesmas features da Célula 1.
============================================================
"""
categorical_cols = [
    "AnimalName",
    "symptoms1",
    "symptoms2",
    "symptoms3",
    "symptoms4",
    "symptoms5",
]

numeric_cols = [
    "unique_symptom_count",
    "unknown_symptom_count",
    "symptom_text_length",
]

feature_cols = categorical_cols + numeric_cols

"""============================================================
6. SEPARAÇÃO TRAIN / VALIDATION / TEST
Utiliza diretamente split_data() da Célula 1.
============================================================
"""

print("\n" + "=" * 70)
print("2. SEPARAÇÃO TRAIN / VALIDATION / TEST")
print("=" * 70)

split = split_data(
    df=df,
    feature_cols=feature_cols,
    target_col=TARGET_NAME,
    validation_size=0.20,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Train : {len(split.x_train)} registros")
print(f"Validation : {len(split.x_val)} registros")
print(f"Test : {len(split.x_test)} registros")

split_summary = {
    "dataset": str(DATASET_PATH),
    "random_state": RANDOM_STATE,
    "train_rows": int(len(split.x_train)),
    "validation_rows": int(len(split.x_val)),
    "test_rows": int(len(split.x_test)),
    "train_positive": int(split.y_train.sum()),
    "validation_positive": int(split.y_val.sum()),
    "test_positive": int(split.y_test.sum()),
    "categorical_columns": categorical_cols,
    "numeric_columns": numeric_cols,
    "feature_columns": feature_cols,
}

with open(
    GENETIC_TABLES_DIR / "split_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

"""============================================================
7. BASELINE
Primeiro executamos os modelos originais da Célula 1.
Isso permite comparar:
BASELINE x ALGORITMO GENÉTICO
============================================================
"""

print("\n" + "=" * 70)
print("3. MODELOS BASELINE")
print("=" * 70)

baseline_models = build_models(
    categorical_cols,
    numeric_cols,
    RANDOM_STATE,
)

baseline_records = []

for model_name, model in baseline_models.items():

    print(f"\nTreinando baseline: {model_name}")

    model.fit(
        split.x_train,
        split.y_train,
    )

    for dataset_name, x_part, y_part in [
        (
            "validation",
            split.x_val,
            split.y_val,
        ),
        (
            "test",
            split.x_test,
            split.y_test,
        ),
    ]:

        metrics = evaluate_model(
            model,
            x_part,
            y_part,
        )

        baseline_records.append(
            {
                "version": "baseline",
                "experiment": "baseline",
                "model": model_name,
                "dataset": dataset_name,
                **metrics,
            }
        )


baseline_metrics = pd.DataFrame(
    baseline_records
)

baseline_metrics.to_csv(
    GENETIC_TABLES_DIR / "baseline_metrics.csv",
    index=False,
    encoding="utf-8",
)

print("\nResultados baseline:")
print(
    baseline_metrics.round(4).to_string(
        index=False
    )
)

"""============================================================
8. ALGORITMO GENÉTICO
Usa diretamente:
GENETIC_SPACES
GENETIC_EXPERIMENTS
genetic_search_model()
definidos na Célula 2.
============================================================
"""

print("\n" + "=" * 70)
print("4. OTIMIZAÇÃO COM ALGORITMO GENÉTICO")
print("=" * 70)

genetic_results = []
genetic_history = []
genetic_models = {}
genetic_parameters = {}

model_names = list(
    GENETIC_SPACES.keys()
)

for experiment_index, experiment in enumerate(
    GENETIC_EXPERIMENTS
):

    print("\n" + "-" * 70)
    print(
        f"{experiment['experiment']}"
    )
    print(
        f"População : {experiment['population_size']}"
    )
    print(
        f"Gerações  : {experiment['generations']}"
    )
    print(
        f"Mutação   : {experiment['mutation_rate']}"
    )
    print("-" * 70)

    for model_index, model_name in enumerate(
        model_names
    ):

        print(
            f"\n>>> Otimizando: {model_name}"
        )

        genetic_seed = (
            RANDOM_STATE
            + experiment_index * 100
            + model_index
        )

        best_model, best_genes, best_cv_metrics, history = (
            genetic_search_model(
                model_name=model_name,
                x_train=split.x_train,
                y_train=split.y_train,
                categorical_cols=categorical_cols,
                numeric_cols=numeric_cols,
                random_state=genetic_seed,
                population_size=experiment[
                    "population_size"
                ],
                generations=experiment[
                    "generations"
                ],
                mutation_rate=experiment[
                    "mutation_rate"
                ],
            )
        )

        experiment_name = experiment[
            "experiment"
        ]

        key = (
            f"{experiment_name}:"
            f"{model_name}"
        )

        genetic_models[key] = best_model

        genetic_parameters[key] = {
            "experiment": experiment_name,
            "model": model_name,
            "genes": best_genes,
            "cv_metrics": best_cv_metrics,
            "random_state": genetic_seed,
        }

        """----------------------------------------------------
        Histórico das gerações
        ----------------------------------------------------"""

        for row in history:

            genetic_history.append(
                row
            )

        """----------------------------------------------------
        Avaliação na validação e teste
        ----------------------------------------------------"""

        for dataset_name, x_part, y_part in [
            (
                "validation",
                split.x_val,
                split.y_val,
            ),
            (
                "test",
                split.x_test,
                split.y_test,
            ),
        ]:

            metrics = evaluate_model(
                best_model,
                x_part,
                y_part,
            )

            genetic_results.append(
                {
                    "version": "genetic",
                    "experiment": experiment_name,
                    "model": model_name,
                    "dataset": dataset_name,
                    "cv_accuracy": best_cv_metrics[
                        "accuracy"
                    ],
                    "cv_recall": best_cv_metrics[
                        "recall"
                    ],
                    "cv_f1": best_cv_metrics[
                        "f1"
                    ],
                    "cv_fitness": best_cv_metrics[
                        "fitness"
                    ],
                    **metrics,
                }
            )

"""============================================================
9. SALVAR RESULTADOS DA BUSCA GENÉTICA
============================================================
"""

genetic_metrics = pd.DataFrame(
    genetic_results
)

genetic_history_df = pd.DataFrame(
    genetic_history
)

genetic_metrics.to_csv(
    GENETIC_TABLES_DIR / "genetic_metrics.csv",
    index=False,
    encoding="utf-8",
)

genetic_history_df.to_csv(
    GENETIC_TABLES_DIR / "genetic_history.csv",
    index=False,
    encoding="utf-8",
)

with open(
    GENETIC_TABLES_DIR / "genetic_parameters.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        genetic_parameters,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("✓ Busca genética concluída.")

"""============================================================
10. COMPARAÇÃO BASELINE x GENÉTICO
============================================================
"""

print("\n" + "=" * 70)
print("5. COMPARAÇÃO BASELINE X GENÉTICO")
print("=" * 70)

comparison_metrics = pd.concat(
    [
        baseline_metrics,
        genetic_metrics,
    ],
    ignore_index=True,
)

comparison_metrics.to_csv(
    GENETIC_TABLES_DIR / "comparison_metrics.csv",
    index=False,
    encoding="utf-8",
)

print(
    comparison_metrics.round(4).to_string(
        index=False
    )
)

"""============================================================
11. SELEÇÃO DO MELHOR MODELO GENÉTICO
REGRA:
1. Recall na validação
2. F1 na validação
3. Accuracy na validação
O TESTE NÃO participa da seleção.
============================================================
"""

validation_results = genetic_metrics[
    genetic_metrics["dataset"] == "validation"
].copy()

best_row = (
    validation_results
    .sort_values(
        [
            "recall_yes",
            "f1_yes",
            "accuracy",
        ],
        ascending=False,
    )
    .iloc[0]
)

best_experiment = best_row[
    "experiment"
]

best_model_name = best_row[
    "model"
]

best_key = (
    f"{best_experiment}:"
    f"{best_model_name}"
)

best_genetic_model = genetic_models[
    best_key
]

best_genetic_parameters = (
    genetic_parameters[
        best_key
    ]
)

print("\n" + "=" * 70)
print("6. MELHOR MODELO GENÉTICO")
print("=" * 70)

print(
    f"Experimento : {best_experiment}"
)

print(
    f"Modelo : {best_model_name}"
)

print(
    f"Recall : {best_row['recall_yes']:.4f}"
)

print(
    f"F1 : {best_row['f1_yes']:.4f}"
)

print(
    f"Accuracy : {best_row['accuracy']:.4f}"
)

print("\nMelhores hiperparâmetros:")

for parameter, value in (
    best_genetic_parameters[
        "genes"
    ].items()
):

    print(
        f"  {parameter}: {value}"
    )

"""============================================================
12. AVALIAÇÃO FINAL DO MELHOR MODELO
Aqui avaliamos:
- treino
- validação
- teste
A avaliação do TESTE ocorre somente depois da escolha do modelo.
============================================================
"""

print("\n" + "=" * 70)
print("7. AVALIAÇÃO FINAL")
print("=" * 70)

best_metrics, classification_reports, final_figures = (
    evaluate_best_model(
        model_name=best_model_name,
        model=best_genetic_model,
        split=split,
        dirs={
            "figures": GENETIC_FIGURES_DIR,
            "tables": GENETIC_TABLES_DIR,
        },
    )
)

print(
    best_metrics.round(4).to_string(
        index=False
    )
)

"""============================================================
13. FEATURE IMPORTANCE
============================================================
"""

print("\n" + "=" * 70)
print("8. FEATURE IMPORTANCE")
print("=" * 70)

feature_importance, importance_figures = (
    model_feature_importance(
        model=best_genetic_model,
        feature_cols=feature_cols,
        split=split,
        dirs={
            "figures": GENETIC_FIGURES_DIR,
            "tables": GENETIC_TABLES_DIR,
        },
        top_n=TOP_N_FEATURES,
    )
)

print(
    feature_importance.head(
        TOP_N_FEATURES
    ).to_string(index=False)
)

"""============================================================
14. SHAP
============================================================
"""

print("\n" + "=" * 70)
print("9. SHAP")
print("=" * 70)

shap_path, shap_message = (
    try_generate_shap(
        model=best_genetic_model,
        split=split,
        feature_cols=feature_cols,
        dirs={
            "figures": GENETIC_FIGURES_DIR,
            "tables": GENETIC_TABLES_DIR,
        },
        top_n=TOP_N_FEATURES,
    )
)

print(shap_message)

"""============================================================
15. EXEMPLO DE PREDIÇÃO
============================================================
"""

prediction_example = (
    write_prediction_example(
        model=best_genetic_model,
        split=split,
        dirs={
            "tables": GENETIC_TABLES_DIR,
        },
    )
)

print("\nExemplo de predição:")

print(
    json.dumps(
        prediction_example,
        indent=2,
        ensure_ascii=False,
    )
)

"""============================================================
16. SALVAR O MODELO GENÉTICO
============================================================
"""

print("\n" + "=" * 70)
print("10. SALVANDO MODELO")
print("=" * 70)

model_path = (
    GENETIC_MODELS_DIR
    / "best_genetic_model.pkl"
)

metadata_path = (
    GENETIC_MODELS_DIR
    / "best_genetic_model_metadata.json"
)

with model_path.open(
    "wb"
) as file:

    pickle.dump(
        best_genetic_model,
        file,
    )


metadata = {
    "dataset": str(DATASET_PATH),
    "model": best_model_name,
    "experiment": best_experiment,
    "random_state": RANDOM_STATE,
    "selection_rule": (
        "Maior recall na validação; "
        "desempate por F1 e accuracy."
    ),
    "hyperparameters": (
        best_genetic_parameters[
            "genes"
        ]
    ),
    "cv_metrics": (
        best_genetic_parameters[
            "cv_metrics"
        ]
    ),
    "validation_metrics": (
        best_row.to_dict()
    ),
    "test_metrics": (
        best_metrics[
            best_metrics["dataset"]
            == "test"
        ].to_dict(
            orient="records"
        )
    ),
    "shap_status": shap_message,
}

with metadata_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )


print(
    f"Modelo salvo em: {model_path}"
)

print(
    f"Metadados salvos em: {metadata_path}"
)

"""============================================================
17. RESUMO FINAL
============================================================
"""

print("\n" + "=" * 70)
print("PIPELINE COMPLETO CONCLUÍDO")
print("=" * 70)

print(
    f"\nMelhor modelo: "
    f"{best_model_name}"
)

print(
    f"Experimento: "
    f"{best_experiment}"
)

print(
    "\nMétricas finais:"
)

print(
    best_metrics.round(4).to_string(
        index=False
    )
)

print(
    "\nPrincipais arquivos:"
)

print(
    f"- Comparação: "
    f"{GENETIC_TABLES_DIR / 'comparison_metrics.csv'}"
)

print(
    f"- Histórico genético: "
    f"{GENETIC_TABLES_DIR / 'genetic_history.csv'}"
)

print(
    f"- Hiperparâmetros: "
    f"{GENETIC_TABLES_DIR / 'genetic_parameters.json'}"
)

print(
    f"- Feature importance: "
    f"{GENETIC_TABLES_DIR / 'feature_importance.csv'}"
)

print(
    f"- Modelo final: "
    f"{model_path}"
)

print(
    f"- Metadados: "
    f"{metadata_path}"
)



✓ Dependências da Célula 1 encontradas.
✓ Dependências da Célula 2 encontradas.

1. CARREGAMENTO E LIMPEZA DOS DADOS
Dataset: /content/data.csv
Linhas originais: 871
Linhas após limpeza: 841
Duplicatas removidas: 28
Distribuição do alvo:
dangerous_binary
1    821
0     20
Name: count, dtype: int64

2. SEPARAÇÃO TRAIN / VALIDATION / TEST
Train : 504 registros
Validation : 168 registros
Test : 169 registros

3. MODELOS BASELINE

Treinando baseline: logistic_regression

Treinando baseline: decision_tree

Treinando baseline: random_forest

Treinando baseline: knn

Resultados baseline:
 version experiment               model    dataset  accuracy  precision_yes  recall_yes  f1_yes  f1_weighted  roc_auc
baseline   baseline logistic_regression validation    0.9702         0.9818      0.9878  0.9848       0.9682   0.9070
baseline   baseline logistic_regression       test    0.9941         0.9940      1.0000  0.9970       0.9937   0.9591
baseline   baseline       decision_tree validation    0.91

                                        feature  importance                    method
                categorical__AnimalName_chicken    0.036722 model_feature_importances
              categorical__AnimalName_buffaloes    0.031849 model_feature_importances
                   numeric__symptom_text_length    0.031326 model_feature_importances
                categorical__symptoms1_weakness    0.029051 model_feature_importances
                categorical__symptoms3_diarrhea    0.027867 model_feature_importances
               categorical__symptoms2_hepatitis    0.027306 model_feature_importances
             categorical__symptoms2_lacrimation    0.027223 model_feature_importances
             categorical__symptoms3_skin rashes    0.023232 model_feature_importances
         categorical__symptoms1_facial swelling    0.022136 model_feature_importances
  categorical__symptoms5_drop on egg production    0.021244 model_feature_importances
             categorical__symptoms3_slow growth    0.0

<Figure size 640x480 with 0 Axes>